# Flipkart Gridlock Training Notebook

In [ ]:

import pandas as pd
import numpy as np

train = pd.read_csv('../dataset/train.csv')
test = pd.read_csv('../dataset/test.csv')

print(train.shape)
print(test.shape)

train.head()


## Feature Engineering

In [ ]:

train['hour'] = train['timestamp'].str.split(':').str[0].astype(int)
train['minute'] = train['timestamp'].str.split(':').str[1].astype(int)

train['slot'] = train['hour'] * 4 + train['minute'] // 15

train.head()


## LightGBM Training

In [ ]:

from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

features = ['slot']

X = train[features]
y = train['demand']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.02,
    num_leaves=63
)

model.fit(X_train, y_train)

pred = model.predict(X_val)

print("R2:", r2_score(y_val, pred))


## Submission Generation

In [ ]:

test['hour'] = test['timestamp'].str.split(':').str[0].astype(int)
test['minute'] = test['timestamp'].str.split(':').str[1].astype(int)

test['slot'] = test['hour'] * 4 + test['minute'] // 15

submission_pred = model.predict(test[features])

submission = pd.DataFrame({
    'Index': range(len(test)),
    'demand': submission_pred
})

submission.to_csv('../outputs/submission_notebook.csv', index=False)

submission.head()
